# Buổi 29 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `hoc_sau.py`; các ô tự dùng bản mới.
Cả notebook chạy khoảng 10–15 phút trên CPU 4 nhân.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu 50 khách hàng (mục 4.1)

In [ ]:
%matplotlib inline
import warnings
from functools import reduce

import hoc_sau as hs

warnings.simplefilter("ignore")
SO_KHACH = 50                      # máy 8 GB RAM: đặt 20
df = hs.doc_dien(so_khach=SO_KHACH)
moc = hs.cac_moc_test(df)
print(df["unique_id"].nunique(), "khách;", len(df.attrs["doi_muc"]), "khách đổi mức:", df.attrs["doi_muc"])
print("mốc test:", moc[0].date(), "→", moc[-1].date(), f"({len(moc)} mốc)")

## Bước 2 — Scaler chỉ học trên train (mục 4.2)

In [ ]:
sc = hs.hoc_scaler(df)
t129 = df[df["unique_id"] == "T129"]
print("scaler của T129 (trung bình, độ lệch chuẩn):", tuple(round(v) for v in sc["T129"]))
print("trung bình T129 trên train (trước MOC_VAL):", round(t129[t129["ds"] < hs.MOC_VAL]["y"].mean()))

## Bước 3 — Cửa sổ chồng lấn và khoảng lạc quan (mục 4.1) — khoảng 1 phút

In [ ]:
nho = hs.chon_it(df)                       # 5 khách đầu, từ 1/11/2013
sc_nho = hs.hoc_scaler(nho)
tr, va = hs.chia_train_val(nho, sc_nho)
te = hs.tap_moc(nho, sc_nho, hs.cac_moc_test(nho))
with hs.so_luong(1):                       # 1 luồng CPU: ra đúng số trong tài liệu (mục 4.6)
    hs.dat_seed(0)
    m = hs.MLP(an=1024, revin=True)
    r = hs.huan_luyen(m, tr, va, so_epoch=200, buoc_moi_epoch=50, kien_nhan=15)
    v, t = hs.mase_tap(m, va, sc_nho, nho), hs.mase_tap(m, te, sc_nho, nho)
print(f"mẫu train {len(tr['X']):,}, val {len(va['X']):,}; epoch val tốt nhất {int(r['lich_su']['val'].idxmin()) + 1}")
print(f"MASE val {v:.3f}   MASE test {t:.3f}")

## Bước 4 — MLP có và không RevIN (mục 4.2–4.3) — khoảng 15 giây

In [ ]:
tr, va = hs.chia_train_val(df, sc)
te = hs.tap_moc(df, sc, moc)
print(f"mẫu train {len(tr['X']):,}, val {len(va['X']):,}, test {len(te['X']):,}")
mang, lich, du_bao = {}, {}, [hs.seasonal_naive(df, moc)]


def hoc(ten, revin):
    nm = ten + ("+RevIN" if revin else "")
    hs.dat_seed(0)                         # seed TRƯỚC khi tạo mạng
    mang[nm] = hs.MO_HINH[ten](revin=revin)
    r = hs.huan_luyen(mang[nm], tr, va)
    lich[nm] = r["lich_su"]
    du_bao.append(hs.du_bao_dl(mang[nm], te, sc, nm)[["unique_id", "moc", "ds", nm]])
    print(f"{nm:12s} {r['giay']:6.1f} giây, {len(r['lich_su'])} epoch, MAE val tốt nhất {r['val_tot_nhat']:.4f}")


hoc("MLP", False)
hoc("MLP", True)
lich["MLP+RevIN"].round(4).iloc[[0, 4, 9, 19, 29]]

## Bước 5 — LSTM và TCN (mục 4.4–4.5) — khoảng 7 phút

In [ ]:
hoc("LSTM", False)
hoc("LSTM", True)
hoc("TCN", True)                           # TCN không RevIN bỏ qua cho nhanh (số có trong tài liệu)

## Bước 6 — So với baseline (mục 4.6) — khoảng 2 phút

In [ ]:
du_bao.append(hs.mstl(df, moc))
du_bao.append(hs.lightgbm(df, moc))
tat_ca = reduce(lambda a, b: a.merge(b, on=["unique_id", "moc", "ds"]), du_bao)
cot = [c for c in tat_ca.columns if c not in ("unique_id", "moc", "ds")]
hs.bang_mase(tat_ca, df, cot, df.attrs["doi_muc"])

## Bước 7 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` → 6/6 xanh.